In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py: Scheme missing.
://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py: Scheme missing.


In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

Q1

In [4]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import llm_structured

In [5]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
load_dotenv()

True

In [7]:
client = OpenAI()

In [8]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
"""

In [9]:
files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]

In [10]:
files_questions = [doc for doc in documents if doc["filename"] in files]

input_tokens_list = []
generated_records = []

In [11]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

for doc in files_questions:
    prompt = {
        "filename": doc["filename"],
        "content": doc["content"],
    }
    user_prompt = json.dumps(prompt)
    result, token_usage = llm_structured(
        client, 
        data_gen_instructions, 
        user_prompt, 
        Questions 
    )
    tokens = token_usage.input_tokens 
    input_tokens_list.append(tokens)

In [12]:
if input_tokens_list:
    avg_input_tokens = sum(input_tokens_list) / len(input_tokens_list)
    print(f"Average input tokens: {avg_input_tokens}")

Average input tokens: 1355.0


Q2

In [13]:
import pandas as pd

df_ground_truth = pd.read_csv('ground-truth.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

def text_search(query, num_results=5):

    return text_index.search(query=query, num_results=num_results)

q = ground_truth[0]["question"]
print(f"Question: {q}")

results = text_search(q)
first_result_filename = results[0]["filename"]

print(f"The filename of the first result is: {first_result_filename}")

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
The filename of the first result is: 01-agentic-rag/lessons/03-rag.md


Q3

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_contents = [chunk["content"] for chunk in chunks]
X = embedding_model.encode(chunk_contents, show_progress_bar=True)

def vector_search(query, num_results=5):
    q_vector = embedding_model.encode(query)
    scores = X.dot(q_vector)
    sorted_indices = np.argsort(scores)[::-1]
    results = []
    for idx in sorted_indices[:num_results]:
        result_chunk = chunks[idx].copy()
        result_chunk["score"] = scores[idx]
        results.append(result_chunk)
        
    return results

q = ground_truth[0]["question"]
print(f"\nEvaluating Question: {q}")

vector_results = vector_search(q, num_results=5)
first_vector_filename = vector_results[0]["filename"]

print(f"The filename of the first vector search result is: {first_vector_filename}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Evaluating Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
The filename of the first vector search result is: 01-agentic-rag/lessons/01-intro.md


Q4

In [17]:
from tqdm.auto import tqdm

In [18]:
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt += 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score += 1.0 / (rank + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        question = q["question"]
        expected_filename = q["filename"]
        results = search_function(question)
        relevance = [res["filename"] == expected_filename for res in results]
        relevance_total.append(relevance)

    return{
        
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    } 

text_metrics = evaluate(ground_truth, text_search)
print(f"mrr: {text_metrics['mrr']}, hit_rate: {text_metrics['hit_rate']}")

  0%|          | 0/360 [00:00<?, ?it/s]

mrr: 0.5942592592592594, hit_rate: 0.7583333333333333


Q5

In [19]:
vector_metrics = evaluate(ground_truth, vector_search)
print(f"mrr: {vector_metrics['mrr']}, hit_rate: {vector_metrics['hit_rate']}")

  0%|          | 0/360 [00:00<?, ?it/s]

mrr: 0.6356944444444446, hit_rate: 0.8083333333333333


Q6

In [20]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0 ))
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

k_values = [1, 50, 100, 200]

for current_k in k_values:
    pinned_hybrid_search = lambda q: hybrid_search(q, k=current_k)

    print(f"\nEvaluating hybrid search with k={current_k}")
    metrics = evaluate(ground_truth, pinned_hybrid_search)

    print(f"MRR: {metrics['mrr']}, Hit Rate: {metrics['hit_rate']}")
    print("-" * 50)


Evaluating hybrid search with k=1


  0%|          | 0/360 [00:00<?, ?it/s]

MRR: 0.6771296296296299, Hit Rate: 0.8611111111111112
--------------------------------------------------

Evaluating hybrid search with k=50


  0%|          | 0/360 [00:00<?, ?it/s]

MRR: 0.6721296296296295, Hit Rate: 0.8472222222222222
--------------------------------------------------

Evaluating hybrid search with k=100


  0%|          | 0/360 [00:00<?, ?it/s]

MRR: 0.6721296296296295, Hit Rate: 0.8472222222222222
--------------------------------------------------

Evaluating hybrid search with k=200


  0%|          | 0/360 [00:00<?, ?it/s]

MRR: 0.6721296296296295, Hit Rate: 0.8472222222222222
--------------------------------------------------
